In [25]:
# --- INSTALAÇÕES (execute se for a primeira vez) ---
# VERSÃO ALTERNATIVA: PyTorch CPU-only (sem precisar do Visual C++)
!pip install pandas torch --index-url https://download.pytorch.org/whl/cpu
!pip install transformers spacy psycopg2-binary python-dotenv
!python -m spacy download pt_core_news_lg

import re
import glob
import json
import pandas as pd
import spacy
import psycopg2
import os
from dotenv import load_dotenv
from transformers import pipeline

# --- CONFIGURAÇÃO DO BANCO DE DADOS ---
# Carrega variáveis de ambiente do arquivo .env no diretório backend
env_path = os.path.join(os.path.dirname(os.getcwd()), 'backend', '.env')
if os.path.exists(env_path):
    load_dotenv(env_path)
    print(f"Arquivo .env carregado de: {env_path}")
else:
    print(f"Arquivo .env não encontrado em: {env_path}")
    print("Configure as variáveis de ambiente manualmente:")

# 🔧 CONFIGURAÇÃO CORRIGIDA: Usa variáveis individuais do .env
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = os.getenv('DB_PORT', '5432')
DB_USER = os.getenv('DB_USER', 'postgres')
DB_PASSWORD = os.getenv('DB_PASSWORD', 'password')
DB_NAME = os.getenv('DB_NAME', 'postgres')

print(f"🔗 Configuração do banco:")
print(f"   Host: {DB_HOST}")
print(f"   Port: {DB_PORT}")
print(f"   User: {DB_USER}")
print(f"   Database: {DB_NAME}")

def get_db_connection():
    """Estabelece conexão com o banco de dados PostgreSQL usando variáveis individuais"""
    try:
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            user=DB_USER,
            password=DB_PASSWORD,
            database=DB_NAME
        )
        return conn
    except Exception as e:
        print(f"Erro ao conectar com o banco: {e}")
        return None

# --- CARREGANDO MODELOS ---
print("Carregando modelo SpaCy (pt_core_news_lg)...")
# Usamos um bloco try-except para lidar com o caso do modelo não estar instalado
try:
    nlp = spacy.load("pt_core_news_lg")
except OSError:
    print("Modelo 'pt_core_news_lg' não encontrado. Baixando e instalando...")
    from spacy.cli import download
    download("pt_core_news_lg")
    nlp = spacy.load("pt_core_news_lg")

print("Carregando modelo de Análise de Sentimento...")
sentiment_classifier = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')

print("Modelos carregados com sucesso!")

# Teste de conexão com o banco
conn = get_db_connection()
if conn:
    print("✅ Conexão com banco estabelecida com sucesso!")
    conn.close()
else:
    print("❌ Falha na conexão com o banco. Verifique as configurações.")

Looking in indexes: https://download.pytorch.org/whl/cpu
     ---------------------------------------- 0.0/568.2 MB ? eta -:--:--
     ---------------------------------------- 0.8/568.2 MB 4.1 MB/s eta 0:02:20
     ---------------------------------------- 1.3/568.2 MB 3.5 MB/s eta 0:02:42
     ---------------------------------------- 1.8/568.2 MB 3.2 MB/s eta 0:02:56
     ---------------------------------------- 2.1/568.2 MB 2.8 MB/s eta 0:03:25
     ---------------------------------------- 2.4/568.2 MB 2.3 MB/s eta 0:04:11
     ---------------------------------------- 2.6/568.2 MB 2.2 MB/s eta 0:04:20
     ---------------------------------------- 3.1/568.2 MB 2.2 MB/s eta 0:04:23
     ---------------------------------------- 3.4/568.2 MB 2.2 MB/s eta 0:04:22
     ---------------------------------------- 3.9/568.2 MB 2.1 MB/s eta 0:04:30
     ---------------------------------------- 4.5/568.2 MB 2.1 MB/s eta 0:04:24
     ---------------------------------------- 4.7/568.2 MB 2.1 MB/s et

Device set to use cpu


Modelos carregados com sucesso!
✅ Conexão com banco estabelecida com sucesso!


In [26]:
# --- FUNÇÕES PARA BUSCAR DADOS DO BANCO ---

def get_chamadas_by_cliente(cliente_id: str) -> list:
    """
    Busca todas as chamadas de um cliente específico no banco de dados.
    
    Args:
        cliente_id (str): ID do cliente
        
    Returns:
        list: Lista de chamadas com transcript_json do cliente
    """
    conn = get_db_connection()
    if not conn:
        print("Erro: Não foi possível conectar ao banco de dados")
        return []
    
    try:
        cursor = conn.cursor()
        # Busca chamadas do cliente que tenham transcript_json
        query = """
        SELECT chamada_id, cliente_id, transcript_json, created_at, nota_qualidade
        FROM chamada 
        WHERE cliente_id = %s 
        AND transcript_json IS NOT NULL
        ORDER BY created_at DESC
        """
        
        cursor.execute(query, (cliente_id,))
        results = cursor.fetchall()
        
        chamadas = []
        for row in results:
            chamada_id, cliente_id, transcript_json, created_at, nota_qualidade = row
            
            # Parse do JSON do transcript
            try:
                if isinstance(transcript_json, str):
                    transcript_data = json.loads(transcript_json)
                else:
                    transcript_data = transcript_json
                
                chamadas.append({
                    'chamada_id': chamada_id,
                    'cliente_id': cliente_id,
                    'transcript_data': transcript_data,
                    'created_at': created_at,
                    'nota_qualidade': nota_qualidade
                })
            except json.JSONDecodeError as e:
                print(f"Erro ao decodificar JSON da chamada {chamada_id}: {e}")
                continue
        
        print(f"Encontradas {len(chamadas)} chamadas para o cliente {cliente_id}")
        return chamadas
        
    except Exception as e:
        print(f"Erro ao buscar chamadas: {e}")
        return []
    finally:
        cursor.close()
        conn.close()

def extract_transcript_text(transcript_data) -> str:
    """
    Extrai o texto da transcrição a partir dos dados do transcript_json.
    
    Args:
        transcript_data: Dados da transcrição (pode ser dict ou string)
        
    Returns:
        str: Texto completo da transcrição
    """
    if isinstance(transcript_data, str):
        return transcript_data
    
    if isinstance(transcript_data, dict):
        # Se for um objeto com estrutura de transcrição
        if 'text' in transcript_data:
            return transcript_data['text']
        
        # Se for uma lista de segmentos/falas
        if 'segments' in transcript_data:
            segments = transcript_data['segments']
            if isinstance(segments, list):
                return ' '.join([seg.get('text', '') for seg in segments if 'text' in seg])
        
        # Se for uma lista direta de falas
        if isinstance(transcript_data, list):
            return ' '.join([item.get('text', '') if isinstance(item, dict) else str(item) for item in transcript_data])
        
        # Tenta converter todo o dict para string
        return str(transcript_data)
    
    return str(transcript_data)

def get_all_clientes() -> list:
    """
    Busca todos os cliente_id únicos que possuem chamadas com transcript.
    
    Returns:
        list: Lista de cliente_ids únicos
    """
    conn = get_db_connection()
    if not conn:
        print("Erro: Não foi possível conectar ao banco de dados")
        return []
    
    try:
        cursor = conn.cursor()
        query = """
        SELECT DISTINCT cliente_id 
        FROM chamada 
        WHERE cliente_id IS NOT NULL 
        AND transcript_json IS NOT NULL
        ORDER BY cliente_id
        """
        
        cursor.execute(query)
        results = cursor.fetchall()
        
        clientes = [row[0] for row in results]
        print(f"Encontrados {len(clientes)} clientes únicos com transcrições")
        return clientes
        
    except Exception as e:
        print(f"Erro ao buscar clientes: {e}")
        return []
    finally:
        cursor.close()
        conn.close()

# Teste das funções
print("Testando conexão e funções do banco...")
clientes = get_all_clientes()
if clientes:
    print(f"Primeiros 5 clientes: {clientes[:5]}")
else:
    print("Nenhum cliente encontrado ou erro na conexão")

Testando conexão e funções do banco...
Encontrados 486 clientes únicos com transcrições
Primeiros 5 clientes: ['1', '10', '100', '101', '102']


In [27]:
# Dicionário principal com a lógica para cada critério de QA
# Agora com categorias e pesos para análise detalhada
QA_CRITERIA = {
    # 1. Abertura / Encerramento
    "1.1_abertura": {
        "keywords": ["comgás", "bom dia", "boa tarde", "boa noite", "meu nome é", "em que posso ajudar"],
        "check_type": "keywords_start", # Verifica no início da transcrição
        "description": "Abertura de contato padrão",
        "categoria": "Abertura/Encerramento",
        "peso": 1.0
    },
    "1.3_encerramento": {
        "keywords": ["algo mais", "posso ajudar em algo mais", "a comgás agradece", "tenha um bom dia", "tenha uma boa tarde"],
        "check_type": "keywords_end", # Verifica no fim da transcrição
        "description": "Encerramento de contato padrão",
        "categoria": "Abertura/Encerramento",
        "peso": 1.0
    },
    "1.2_espera": {
        "keywords": ["só um momento", "aguarde um instante", "só um minuto", "vou verificar", "obrigado por aguardar"],
        "check_type": "keywords_any", # Verifica em qualquer lugar
        "description": "Sinalizou procedimento de espera",
        "categoria": "Abertura/Encerramento",
        "peso": 0.8
    },

    # 2. Condução do Atendimento
    "2.1_empatia": {
        "keywords": ["eu entendo", "compreendo", "sinto muito", "peço desculpas pelo transtorno", "vamos resolver"],
        "check_type": "keywords_any",
        "description": "Demonstrou empatia",
        "categoria": "Condução do Atendimento",
        "peso": 1.2
    },
    "2.2_cordialidade": {
        "keywords": ["por favor", "por gentileza", "obrigado", "obrigada", "senhor", "senhora"],
        "check_type": "keywords_any",
        "description": "Tratou o cliente com cordialidade",
        "categoria": "Condução do Atendimento",
        "peso": 1.0
    },
    "2.3_dialogo_vicios": {
        "keywords": ["tipo assim", "né", "tá", "mano", "cara", "veio"],
        "check_type": "negative_keywords", # Critério reprova se encontrar essas palavras
        "description": "Diálogo sem vícios de linguagem/gírias",
        "categoria": "Condução do Atendimento",
        "peso": 1.0
    },

    # 3. Processos e Procedimentos
    "3.0_identificacao_positiva": {
        "keywords": [r"confirmar seu (cpf|nome|endereço)", r"qual (seu|o) (cpf|nome completo)", r"código de usuário"],
        "check_type": "regex_any",
        "description": "Realizou a identificação positiva",
        "categoria": "Processos e Procedimentos",
        "peso": 1.5
    },
    "3.4_oportunidades": {
        "keywords": ["comgás virtual", "fatura digital", "conta por e-mail", "débito automático", "aproveitando o contato"],
        "check_type": "keywords_any",
        "description": "Criou oportunidades de negócio / fomentou autosserviço",
        "categoria": "Processos e Procedimentos",
        "peso": 0.8
    },
    "3.5_pesquisa_satisfacao": {
        "keywords": ["pesquisa de satisfação"],
        "check_type": "keywords_end",
        "description": "Direcionou para a pesquisa de satisfação",
        "categoria": "Processos e Procedimentos",
        "peso": 0.7
    },

    # 4. NCG / Falhas Graves
    "4.0_etica": {
        "keywords": ["absurdo", "incompetente", "esse sistema", "outro atendente", "fez errado", "droga", "inferno"],
        "check_type": "negative_keywords",
        "description": "Ética (não difamou a empresa ou colegas)",
        "categoria": "NCG / Falhas Graves",
        "peso": 2.0
    },
    "4.4_abertura_os": {
        "keywords": [r"(gerar|abrir|registrar) (a os|uma os|ordem de serviço)", r"número da sua os", r"protocolo da os"],
        "check_type": "regex_any",
        "description": "Mencionou a abertura ou o registro de uma OS",
        "categoria": "NCG / Falhas Graves",
        "peso": 1.3
    },
    "4.5_desconexao_indevida": {
        "keywords": ["alô", "está me ouvindo", "a ligação caiu"],
        "check_type": "keywords_end", # Se o cliente diz isso no final, pode ser um sinal
        "description": "Indício de desconexão indevida (EXPERIMENTAL)",
        "categoria": "NCG / Falhas Graves",
        "peso": 1.8
    }
}

# Adicionamos aqui critérios que são difíceis ou impossíveis de automatizar apenas com a transcrição.
# O script irá marcá-los para revisão manual.
MANUAL_REVIEW_CRITERIA = [
    "2.0", "3.1", "3.2", "3.3", "4.1", "4.2", "4.3", "4.6", "4.7", "4.8", "4.10"
]

print(f"📋 Configurados {len(QA_CRITERIA)} critérios de avaliação com categorias e pesos")
print("🏷️  Categorias disponíveis:")
categorias = set(c["categoria"] for c in QA_CRITERIA.values())
for i, cat in enumerate(sorted(categorias), 1):
    total_peso = sum(c["peso"] for c in QA_CRITERIA.values() if c["categoria"] == cat)
    criterios_count = len([c for c in QA_CRITERIA.values() if c["categoria"] == cat])
    print(f"   {i}. {cat}: {criterios_count} critérios (peso total: {total_peso})")

📋 Configurados 12 critérios de avaliação com categorias e pesos
🏷️  Categorias disponíveis:
   1. Abertura/Encerramento: 3 critérios (peso total: 2.8)
   2. Condução do Atendimento: 3 critérios (peso total: 3.2)
   3. NCG / Falhas Graves: 3 critérios (peso total: 5.1)
   4. Processos e Procedimentos: 3 critérios (peso total: 3.0)


In [28]:
def evaluate_call_quality(transcript: str, chamada_info: dict = None) -> dict:
    """
    Avalia uma única transcrição com base nos critérios definidos em QA_CRITERIA.

    Args:
        transcript (str): Texto da transcrição da chamada
        chamada_info (dict): Informações adicionais da chamada (opcional)

    Returns:
        Um dicionário com o resultado ('Aprovado' ou 'Reprovado') para cada critério.
    """
    results = {}
    transcript_lower = transcript.lower()
    transcript_len = len(transcript_lower)

    # Define seções da transcrição (início e fim representam ~15% cada)
    start_slice = transcript_lower[:int(transcript_len * 0.15)]
    end_slice = transcript_lower[-int(transcript_len * 0.15):]

    for key, criteria in QA_CRITERIA.items():
        check_type = criteria["check_type"]
        keywords = criteria["keywords"]
        found = False

        # Define qual parte da transcrição analisar
        target_text = transcript_lower
        if check_type == "keywords_start":
            target_text = start_slice
        elif check_type == "keywords_end":
            target_text = end_slice

        # Verifica se as palavras-chave foram encontradas
        if "keywords" in check_type:
            if any(keyword in target_text for keyword in keywords):
                found = True
        elif "regex" in check_type:
            if any(re.search(pattern, target_text, re.IGNORECASE) for pattern in keywords):
                found = True

        # Aplica a lógica de aprovação/reprovação
        if check_type == "negative_keywords":
            results[key] = "Reprovado" if found else "Aprovado"
        else:
            results[key] = "Aprovado" if found else "Reprovado"

    # Adiciona informações extras se disponíveis
    if chamada_info:
        results['_metadata'] = {
            'chamada_id': chamada_info.get('chamada_id'),
            'cliente_id': chamada_info.get('cliente_id'),
            'created_at': chamada_info.get('created_at'),
            'nota_qualidade_original': chamada_info.get('nota_qualidade')
        }

    return results

def evaluate_call_quality_detailed(transcript: str, chamada_info: dict = None) -> dict:
    """
    NOVA FUNÇÃO: Avalia uma transcrição e retorna resultado detalhado para salvar no banco.
    
    Returns:
        dict: Estrutura completa com metadados e resultados por critério
    """
    from datetime import datetime
    
    transcript_lower = transcript.lower()
    transcript_len = len(transcript_lower)

    # Define seções da transcrição (início e fim representam ~15% cada)
    start_slice = transcript_lower[:int(transcript_len * 0.15)]
    end_slice = transcript_lower[-int(transcript_len * 0.15):]

    criterios_avaliados = {}
    
    for key, criteria in QA_CRITERIA.items():
        check_type = criteria["check_type"]
        keywords = criteria["keywords"]
        found = False
        evidencias = []

        # Define qual parte da transcrição analisar
        target_text = transcript_lower
        if check_type == "keywords_start":
            target_text = start_slice
        elif check_type == "keywords_end":
            target_text = end_slice

        # Verifica se as palavras-chave foram encontradas e coleta evidências
        if "keywords" in check_type:
            for keyword in keywords:
                if keyword in target_text:
                    found = True
                    evidencias.append(keyword)
                    
        elif "regex" in check_type:
            import re
            for pattern in keywords:
                matches = re.finditer(pattern, target_text, re.IGNORECASE)
                for match in matches:
                    found = True
                    evidencias.append(match.group())

        # Aplica a lógica de aprovação/reprovação
        if check_type == "negative_keywords":
            resultado = "Reprovado" if found else "Aprovado"
            # Para critérios negativos, evidência indica problema
            evidencias_problema = evidencias if found else []
        else:
            resultado = "Aprovado" if found else "Reprovado"
            evidencias_problema = [] if found else ["Critério não identificado na transcrição"]

        criterios_avaliados[key] = {
            "codigo": key,
            "descricao": criteria["description"],
            "categoria": criteria.get("categoria", "Geral"),
            "resultado": resultado,
            "peso": criteria.get("peso", 1.0),
            "evidencias_encontradas": evidencias,
            "observacoes": evidencias_problema
        }

    # Calcula estatísticas gerais
    total_criterios = len(criterios_avaliados)
    criterios_aprovados = sum(1 for c in criterios_avaliados.values() if c["resultado"] == "Aprovado")
    
    # Calcula score ponderado
    peso_total = sum(criteria.get("peso", 1.0) for criteria in QA_CRITERIA.values())
    pontos_obtidos = sum(
        criterio["peso"] for criterio in criterios_avaliados.values() 
        if criterio["resultado"] == "Aprovado"
    )
    score_ponderado = pontos_obtidos / peso_total if peso_total > 0 else 0
    
    # Calcula score simples
    score_simples = criterios_aprovados / total_criterios if total_criterios > 0 else 0

    # Estrutura final do JSON
    avaliacao_completa = {
        "metadata": {
            "chamada_id": chamada_info.get('chamada_id') if chamada_info else None,
            "cliente_id": chamada_info.get('cliente_id') if chamada_info else None,
            "data_avaliacao": datetime.now().isoformat(),
            "versao_algoritmo": "v2.0_com_json_detalhado",
            "tamanho_transcricao": len(transcript),
            "nota_qualidade_anterior": chamada_info.get('nota_qualidade') if chamada_info else None
        },
        "resumo": {
            "total_criterios": total_criterios,
            "criterios_aprovados": criterios_aprovados,
            "criterios_reprovados": total_criterios - criterios_aprovados,
            "score_simples": round(score_simples, 3),
            "score_ponderado": round(score_ponderado, 3),
            "nota_calculada_1_10": max(1, min(10, round(1 + (score_ponderado * 9))))
        },
        "criterios": criterios_avaliados,
        "estatisticas_por_categoria": {}
    }
    
    # Calcula estatísticas por categoria
    categorias = {}
    for criterio in criterios_avaliados.values():
        cat = criterio["categoria"]
        if cat not in categorias:
            categorias[cat] = {"total": 0, "aprovados": 0, "peso_total": 0, "pontos_obtidos": 0}
        
        categorias[cat]["total"] += 1
        categorias[cat]["peso_total"] += criterio["peso"]
        
        if criterio["resultado"] == "Aprovado":
            categorias[cat]["aprovados"] += 1
            categorias[cat]["pontos_obtidos"] += criterio["peso"]
    
    for cat, stats in categorias.items():
        stats["taxa_aprovacao"] = stats["aprovados"] / stats["total"] if stats["total"] > 0 else 0
        stats["score_ponderado"] = stats["pontos_obtidos"] / stats["peso_total"] if stats["peso_total"] > 0 else 0
    
    avaliacao_completa["estatisticas_por_categoria"] = categorias
    
    return avaliacao_completa

def analyze_cliente_quality(cliente_id: str) -> dict:
    """
    Analisa todas as chamadas de um cliente e gera relatório de qualidade.
    
    Args:
        cliente_id (str): ID do cliente a ser analisado
        
    Returns:
        dict: Relatório completo de análise do cliente
    """
    print(f"Analisando chamadas do cliente: {cliente_id}")
    
    # Busca todas as chamadas do cliente
    chamadas = get_chamadas_by_cliente(cliente_id)
    
    if not chamadas:
        return {
            'cliente_id': cliente_id,
            'error': 'Nenhuma chamada encontrada para este cliente'
        }
    
    analysis_results = []
    criteria_summary = {}
    
    # Analisa cada chamada individualmente
    for chamada in chamadas:
        transcript_text = extract_transcript_text(chamada['transcript_data'])
        
        if not transcript_text or len(transcript_text.strip()) < 10:
            print(f"Transcrição vazia ou muito curta para chamada {chamada['chamada_id']}")
            continue
        
        # Avalia a qualidade da chamada
        quality_result = evaluate_call_quality(transcript_text, chamada)
        
        analysis_results.append({
            'chamada_id': chamada['chamada_id'],
            'created_at': chamada['created_at'],
            'nota_qualidade_original': chamada['nota_qualidade'],
            'transcript_length': len(transcript_text),
            'quality_analysis': quality_result
        })
        
        # Agrega os resultados por critério
        for criterion, result in quality_result.items():
            if criterion.startswith('_'):  # Pula metadados
                continue
                
            if criterion not in criteria_summary:
                criteria_summary[criterion] = {'Aprovado': 0, 'Reprovado': 0, 'total': 0}
            
            criteria_summary[criterion][result] += 1
            criteria_summary[criterion]['total'] += 1
    
    # Calcula estatísticas finais
    total_calls = len(analysis_results)
    overall_score = 0
    
    for criterion, stats in criteria_summary.items():
        if stats['total'] > 0:
            approval_rate = stats['Aprovado'] / stats['total']
            stats['approval_rate'] = approval_rate
            overall_score += approval_rate
    
    overall_score = overall_score / len(criteria_summary) if criteria_summary else 0
    
    return {
        'cliente_id': cliente_id,
        'total_calls_analyzed': total_calls,
        'overall_quality_score': overall_score,
        'criteria_summary': criteria_summary,
        'detailed_results': analysis_results,
        'recommendations': generate_recommendations(criteria_summary)
    }

def generate_recommendations(criteria_summary: dict) -> list:
    """
    Gera recomendações baseadas nos resultados da análise.
    
    Args:
        criteria_summary (dict): Resumo dos critérios analisados
        
    Returns:
        list: Lista de recomendações
    """
    recommendations = []
    
    for criterion, stats in criteria_summary.items():
        if stats['total'] > 0 and stats['approval_rate'] < 0.7:  # Menos de 70% de aprovação
            criterion_info = QA_CRITERIA.get(criterion, {})
            description = criterion_info.get('description', criterion)
            
            recommendations.append({
                'criterion': criterion,
                'description': description,
                'approval_rate': stats['approval_rate'],
                'suggestion': f"Melhorar {description.lower()} - apenas {stats['approval_rate']:.1%} das chamadas atendem este critério"
            })
    
    return recommendations

# Função utilitária para exibir resultados de forma organizada
def print_analysis_summary(analysis_result: dict):
    """
    Exibe um resumo organizado da análise de qualidade.
    """
    if 'error' in analysis_result:
        print(f"❌ Erro: {analysis_result['error']}")
        return
    
    cliente_id = analysis_result['cliente_id']
    total_calls = analysis_result['total_calls_analyzed']
    overall_score = analysis_result['overall_quality_score']
    
    print(f"\n📊 RELATÓRIO DE QUALIDADE - Cliente: {cliente_id}")
    print(f"📞 Total de chamadas analisadas: {total_calls}")
    print(f"⭐ Score geral de qualidade: {overall_score:.1%}")
    print(f"{'='*50}")
    
    # Mostra resumo por critério
    print("\n📋 RESUMO POR CRITÉRIO:")
    for criterion, stats in analysis_result['criteria_summary'].items():
        criterion_info = QA_CRITERIA.get(criterion, {})
        description = criterion_info.get('description', criterion)
        approval_rate = stats['approval_rate']
        
        status_emoji = "✅" if approval_rate >= 0.8 else "⚠️" if approval_rate >= 0.6 else "❌"
        print(f"{status_emoji} {description}: {approval_rate:.1%} ({stats['Aprovado']}/{stats['total']})")
    
    # Mostra recomendações
    recommendations = analysis_result['recommendations']
    if recommendations:
        print(f"\n🎯 RECOMENDAÇÕES ({len(recommendations)} pontos de melhoria):")
        for i, rec in enumerate(recommendations, 1):
            print(f"{i}. {rec['suggestion']}")
    else:
        print(f"\n🎉 Excelente! Todos os critérios atingiram pelo menos 70% de aprovação.")

print("✅ Funções de análise atualizadas com suporte a JSON detalhado!")
print("📋 Nova função disponível: evaluate_call_quality_detailed()")

✅ Funções de análise atualizadas com suporte a JSON detalhado!
📋 Nova função disponível: evaluate_call_quality_detailed()


In [29]:
# --- EXEMPLOS DE USO ---

# 1. LISTAR TODOS OS CLIENTES DISPONÍVEIS
print("🔍 Buscando clientes disponíveis...")
clientes_disponiveis = get_all_clientes()

if clientes_disponiveis:
    print(f"Total de clientes encontrados: {len(clientes_disponiveis)}")
    print("Primeiros 10 clientes:")
    for i, cliente in enumerate(clientes_disponiveis[:10], 1):
        print(f"  {i}. {cliente}")
else:
    print("❌ Nenhum cliente encontrado. Verifique:")
    print("  - Se o banco está rodando")
    print("  - Se a variável DATABASE_URL está correta")
    print("  - Se há dados na tabela 'chamada'")

print("\n" + "="*60)

# 2. ANÁLISE DE UM CLIENTE ESPECÍFICO (exemplo)
if clientes_disponiveis:
    # Pega o primeiro cliente como exemplo
    cliente_exemplo = clientes_disponiveis[0]
    
    print(f"📊 EXEMPLO: Analisando cliente {cliente_exemplo}")
    print("-" * 40)
    
    # Executa a análise
    resultado_analise = analyze_cliente_quality(cliente_exemplo)
    
    # Exibe o relatório
    print_analysis_summary(resultado_analise)
    
    print("\n" + "="*60)
    print("💡 COMO USAR:")
    print("Para analisar um cliente específico, use:")
    print("  resultado = analyze_cliente_quality('SEU_CLIENTE_ID')")
    print("  print_analysis_summary(resultado)")
    
else:
    print("⚠️ Não foi possível executar exemplo - nenhum cliente encontrado")

🔍 Buscando clientes disponíveis...
Encontrados 486 clientes únicos com transcrições
Total de clientes encontrados: 486
Primeiros 10 clientes:
  1. 1
  2. 10
  3. 100
  4. 101
  5. 102
  6. 103
  7. 104
  8. 105
  9. 106
  10. 107

📊 EXEMPLO: Analisando cliente 1
----------------------------------------
Analisando chamadas do cliente: 1
Encontradas 1 chamadas para o cliente 1

📊 RELATÓRIO DE QUALIDADE - Cliente: 1
📞 Total de chamadas analisadas: 1
⭐ Score geral de qualidade: 41.7%

📋 RESUMO POR CRITÉRIO:
❌ Abertura de contato padrão: 0.0% (0/1)
❌ Encerramento de contato padrão: 0.0% (0/1)
✅ Sinalizou procedimento de espera: 100.0% (1/1)
✅ Demonstrou empatia: 100.0% (1/1)
✅ Tratou o cliente com cordialidade: 100.0% (1/1)
❌ Diálogo sem vícios de linguagem/gírias: 0.0% (0/1)
❌ Realizou a identificação positiva: 0.0% (0/1)
❌ Criou oportunidades de negócio / fomentou autosserviço: 0.0% (0/1)
❌ Direcionou para a pesquisa de satisfação: 0.0% (0/1)
✅ Ética (não difamou a empresa ou colegas): 10

In [30]:
# 🎯 ANÁLISE DE CHAMADA ESPECÍFICA POR ID
# ==========================================
# 📝 INSTRUÇÃO: Altere apenas o valor da variável CHAMADA_ID abaixo
# ==========================================

CHAMADA_ID = 110  # ⬅️ COLOQUE AQUI O ID DA CHAMADA QUE VOCÊ QUER ANALISAR

def analyze_single_call(chamada_id: int) -> dict:
    """
    Analisa uma única chamada específica pelo ID.
    
    Args:
        chamada_id (int): ID da chamada a ser analisada
        
    Returns:
        dict: Resultado da análise da chamada
    """
    conn = get_db_connection()
    if not conn:
        return {'error': 'Não foi possível conectar ao banco de dados'}
    
    try:
        cursor = conn.cursor()
        # Busca a chamada específica
        query = """
        SELECT chamada_id, cliente_id, transcript_json, created_at, nota_qualidade
        FROM chamada 
        WHERE chamada_id = %s
        """
        
        cursor.execute(query, (chamada_id,))
        result = cursor.fetchone()
        
        if not result:
            return {'error': f'Chamada {chamada_id} não encontrada'}
        
        chamada_id, cliente_id, transcript_json, created_at, nota_qualidade = result
        
        # Parse do JSON do transcript
        try:
            if isinstance(transcript_json, str):
                transcript_data = json.loads(transcript_json)
            else:
                transcript_data = transcript_json
        except (json.JSONDecodeError, TypeError):
            return {'error': f'Erro ao decodificar transcript_json da chamada {chamada_id}'}
        
        # Extrai o texto da transcrição
        transcript_text = extract_transcript_text(transcript_data)
        
        if not transcript_text or len(transcript_text.strip()) < 10:
            return {'error': f'Transcrição vazia ou muito curta para chamada {chamada_id}'}
        
        # Informações da chamada
        chamada_info = {
            'chamada_id': chamada_id,
            'cliente_id': cliente_id,
            'created_at': created_at,
            'nota_qualidade': nota_qualidade
        }
        
        # Avalia a qualidade da chamada
        quality_result = evaluate_call_quality(transcript_text, chamada_info)
        
        # Calcula score geral
        total_criteria = 0
        approved_criteria = 0
        
        for criterion, result in quality_result.items():
            if criterion.startswith('_'):  # Pula metadados
                continue
            total_criteria += 1
            if result == 'Aprovado':
                approved_criteria += 1
        
        overall_score = approved_criteria / total_criteria if total_criteria > 0 else 0
        
        return {
            'chamada_id': chamada_id,
            'cliente_id': cliente_id,
            'created_at': created_at,
            'nota_qualidade_original': nota_qualidade,
            'transcript_length': len(transcript_text),
            'transcript_preview': transcript_text[:200] + "..." if len(transcript_text) > 200 else transcript_text,
            'overall_score': overall_score,
            'quality_analysis': quality_result,
            'approved_count': approved_criteria,
            'total_criteria': total_criteria
        }
        
    except Exception as e:
        return {'error': f'Erro ao analisar chamada: {e}'}
    finally:
        cursor.close()
        conn.close()

def print_single_call_analysis(result: dict):
    """
    Exibe resultado da análise de uma chamada específica.
    """
    if 'error' in result:
        print(f"❌ Erro: {result['error']}")
        return
    
    print(f"\n🔍 ANÁLISE DETALHADA DA CHAMADA")
    print(f"{'='*60}")
    print(f"📞 ID da Chamada: {result['chamada_id']}")
    print(f"👤 Cliente ID: {result['cliente_id']}")
    print(f"📅 Data/Hora: {result['created_at']}")
    print(f"⭐ Nota Original: {result['nota_qualidade_original']}")
    print(f"📏 Tamanho da Transcrição: {result['transcript_length']} caracteres")
    print(f"🎯 Score de Qualidade: {result['overall_score']:.1%} ({result['approved_count']}/{result['total_criteria']})")
    
    print(f"\n📝 PRÉVIA DA TRANSCRIÇÃO:")
    print(f"{'─'*40}")
    print(result['transcript_preview'])
    
    print(f"\n📋 AVALIAÇÃO POR CRITÉRIO:")
    print(f"{'─'*40}")
    
    for criterion, result_val in result['quality_analysis'].items():
        if criterion.startswith('_'):  # Pula metadados
            continue
            
        criterion_info = QA_CRITERIA.get(criterion, {})
        description = criterion_info.get('description', criterion)
        
        status_emoji = "✅" if result_val == "Aprovado" else "❌"
        print(f"{status_emoji} {description}: {result_val}")

# EXECUÇÃO DA ANÁLISE
print(f"🔍 Analisando chamada ID: {CHAMADA_ID}")
print("─" * 50)

resultado = analyze_single_call(CHAMADA_ID)
print_single_call_analysis(resultado)

print(f"\n💡 DICA: Para analisar outra chamada, mude o valor de CHAMADA_ID no início desta célula e execute novamente.")

🔍 Analisando chamada ID: 110
──────────────────────────────────────────────────

🔍 ANÁLISE DETALHADA DA CHAMADA
📞 ID da Chamada: 110
👤 Cliente ID: 110
📅 Data/Hora: 2025-09-12 13:48:43.728636
⭐ Nota Original: 3
📏 Tamanho da Transcrição: 24074 caracteres
🎯 Score de Qualidade: 50.0% (6/12)

📝 PRÉVIA DA TRANSCRIÇÃO:
────────────────────────────────────────
{'meta': {'seed': 447785658, 'persona': 'irritado', 'scenario': 'Segunda via não chega e discussão sobre vencimento e multa', 'complyCriteriaIds': ['3.4', '1.3', '1.1', '4.8', '2.3', '4.3', '4.4', '1....

📋 AVALIAÇÃO POR CRITÉRIO:
────────────────────────────────────────
❌ Abertura de contato padrão: Reprovado
✅ Encerramento de contato padrão: Aprovado
✅ Sinalizou procedimento de espera: Aprovado
✅ Demonstrou empatia: Aprovado
✅ Tratou o cliente com cordialidade: Aprovado
❌ Diálogo sem vícios de linguagem/gírias: Reprovado
✅ Realizou a identificação positiva: Aprovado
❌ Criou oportunidades de negócio / fomentou autosserviço: Reprovado
❌ 

In [35]:
# DIAGNOSTICO: POR QUE ALGUMAS CHAMADAS NAO FORAM PROCESSADAS?
# ==============================================================

def diagnose_processing_issues():
    """
    Analisa as chamadas que nao foram processadas e identifica os motivos.
    """
    conn = get_db_connection()
    if not conn:
        print("Erro: Nao foi possivel conectar ao banco de dados")
        return
    
    try:
        cursor = conn.cursor()
        
        print("=== DIAGNOSTICO DE PROCESSAMENTO ===\n")
        
        # 1. Total de chamadas na tabela
        cursor.execute("SELECT COUNT(*) FROM chamada")
        total_chamadas = cursor.fetchone()[0]
        print(f"1. TOTAL DE CHAMADAS NA TABELA: {total_chamadas}")
        
        # 2. Chamadas com transcript_json
        cursor.execute("SELECT COUNT(*) FROM chamada WHERE transcript_json IS NOT NULL")
        com_transcript = cursor.fetchone()[0]
        print(f"2. CHAMADAS COM TRANSCRIPT_JSON: {com_transcript}")
        
        # 3. Chamadas com transcript_json vazio ou null (CORRIGIDO)
        cursor.execute("""
        SELECT COUNT(*) FROM chamada 
        WHERE transcript_json IS NULL 
           OR transcript_json::text = '' 
           OR transcript_json::text = 'null'
           OR transcript_json::text = '{}'
        """)
        sem_transcript = cursor.fetchone()[0]
        print(f"3. CHAMADAS SEM TRANSCRIPT VALIDO: {sem_transcript}")
        
        # 4. Chamadas com nota_qualidade atualizada
        cursor.execute("SELECT COUNT(*) FROM chamada WHERE nota_qualidade IS NOT NULL")
        com_nota = cursor.fetchone()[0]
        print(f"4. CHAMADAS COM NOTA_QUALIDADE: {com_nota}")
        
        # 5. Chamadas com avaliacao_json
        cursor.execute("SELECT COUNT(*) FROM chamada WHERE avaliacao_json IS NOT NULL")
        com_avaliacao = cursor.fetchone()[0]
        print(f"5. CHAMADAS COM AVALIACAO_JSON: {com_avaliacao}")
        
        print(f"\n=== ANALISE ===")
        print(f"• Chamadas que deveriam ser processadas: {com_transcript}")
        print(f"• Chamadas efetivamente processadas: {com_avaliacao}")
        print(f"• Chamadas nao processadas: {com_transcript - com_avaliacao}")
        
        if com_transcript > com_avaliacao:
            print(f"\n=== INVESTIGANDO CHAMADAS NAO PROCESSADAS ===")
            
            # 6. Mostra exemplos de transcript_json problematicos
            cursor.execute("""
            SELECT chamada_id, cliente_id, 
                   CASE 
                       WHEN transcript_json IS NULL THEN 'NULL'
                       WHEN transcript_json::text = '' THEN 'VAZIO'
                       WHEN transcript_json::text = '{}' THEN 'OBJETO_VAZIO'
                       WHEN transcript_json::text = 'null' THEN 'NULL_STRING'
                       ELSE 'OUTRO'
                   END as tipo_problema,
                   LENGTH(transcript_json::text) as tamanho,
                   LEFT(transcript_json::text, 100) as amostra
            FROM chamada 
            WHERE transcript_json IS NOT NULL 
            AND avaliacao_json IS NULL
            LIMIT 10
            """)
            
            problematicas = cursor.fetchall()
            
            if problematicas:
                print(f"\nEXEMPLOS DE CHAMADAS NAO PROCESSADAS:")
                print("-" * 80)
                for row in problematicas:
                    chamada_id, cliente_id, tipo, tamanho, amostra = row
                    print(f"• Chamada {chamada_id} (Cliente: {cliente_id})")
                    print(f"  Tipo: {tipo} | Tamanho: {tamanho} chars")
                    print(f"  Amostra: {amostra}...")
                    print()
            
            # 7. Tenta processar uma chamada problematica manualmente
            print("=== TESTE MANUAL DE UMA CHAMADA PROBLEMATICA ===")
            if problematicas:
                teste_id = problematicas[0][0]
                print(f"Testando chamada ID: {teste_id}")
                
                cursor.execute("""
                SELECT transcript_json FROM chamada WHERE chamada_id = %s
                """, (teste_id,))
                
                transcript_raw = cursor.fetchone()[0]
                print(f"Transcript bruto (tipo: {type(transcript_raw)}):")
                print(f"Conteudo: {transcript_raw}")
                
                try:
                    # Tenta extrair texto
                    transcript_text = extract_transcript_text(transcript_raw)
                    print(f"Texto extraido (tamanho: {len(transcript_text)}): {transcript_text[:200]}...")
                    
                    if len(transcript_text.strip()) < 10:
                        print("PROBLEMA: Transcricao muito curta após extracao!")
                    else:
                        print("Transcricao valida - deveria ter sido processada.")
                        
                        # Tenta processar esta chamada manualmente
                        print("TENTANDO PROCESSAR MANUALMENTE...")
                        try:
                            chamada_info = {
                                'chamada_id': teste_id,
                                'cliente_id': cliente_id,
                                'created_at': None,
                                'nota_qualidade': None
                            }
                            
                            avaliacao = evaluate_call_quality_detailed(transcript_text, chamada_info)
                            print(f"SUCESSO! Processamento manual funcionou.")
                            print(f"Nota calculada: {avaliacao['resumo']['nota_calculada_1_10']}")
                            print(f"Score: {avaliacao['resumo']['score_ponderado']:.1%}")
                            
                        except Exception as process_error:
                            print(f"ERRO no processamento manual: {process_error}")
                        
                except Exception as e:
                    print(f"ERRO na extracao de texto: {e}")
        
        # 8. Verifica se ha erros de encoding ou formato JSON (CORRIGIDO)
        print(f"\n=== VERIFICACAO DE FORMATOS JSON ===")
        cursor.execute("""
        SELECT 
               CASE 
                   WHEN transcript_json::text ~ '^\\{.*\\}$' THEN 'JSON_OBJECT'
                   WHEN transcript_json::text ~ '^\\[.*\\]$' THEN 'JSON_ARRAY'
                   WHEN transcript_json::text ~ '^".*"$' THEN 'JSON_STRING'
                   ELSE 'FORMATO_DESCONHECIDO'
               END as formato_json,
               COUNT(*) as quantidade
        FROM chamada 
        WHERE transcript_json IS NOT NULL
        GROUP BY formato_json
        ORDER BY COUNT(*) DESC
        """)
        
        formatos = cursor.fetchall()
        print("Distribuicao de formatos JSON:")
        for formato, quantidade in formatos:
            print(f"• {formato}: {quantidade} chamadas")
        
        # 9. Verifica tamanhos de transcript muito pequenos
        print(f"\n=== VERIFICACAO DE TAMANHOS ===")
        cursor.execute("""
        SELECT 
            CASE 
                WHEN LENGTH(transcript_json::text) < 10 THEN 'MUITO_PEQUENO'
                WHEN LENGTH(transcript_json::text) < 50 THEN 'PEQUENO' 
                WHEN LENGTH(transcript_json::text) < 200 THEN 'MEDIO'
                ELSE 'GRANDE'
            END as tamanho_categoria,
            COUNT(*) as quantidade
        FROM chamada 
        WHERE transcript_json IS NOT NULL
        GROUP BY tamanho_categoria
        ORDER BY quantidade DESC
        """)
        
        tamanhos = cursor.fetchall()
        print("Distribuicao por tamanho de transcript:")
        for categoria, quantidade in tamanhos:
            print(f"• {categoria}: {quantidade} chamadas")
        
    except Exception as e:
        print(f"Erro no diagnostico: {e}")
        import traceback
        traceback.print_exc()
    finally:
        cursor.close()
        conn.close()

# Executa o diagnostico
diagnose_processing_issues()

=== DIAGNOSTICO DE PROCESSAMENTO ===

1. TOTAL DE CHAMADAS NA TABELA: 486
2. CHAMADAS COM TRANSCRIPT_JSON: 486
3. CHAMADAS SEM TRANSCRIPT VALIDO: 0
4. CHAMADAS COM NOTA_QUALIDADE: 486
5. CHAMADAS COM AVALIACAO_JSON: 444

=== ANALISE ===
• Chamadas que deveriam ser processadas: 486
• Chamadas efetivamente processadas: 444
• Chamadas nao processadas: 42

=== INVESTIGANDO CHAMADAS NAO PROCESSADAS ===

EXEMPLOS DE CHAMADAS NAO PROCESSADAS:
--------------------------------------------------------------------------------
• Chamada 91 (Cliente: 91)
  Tipo: OUTRO | Tamanho: 16021 chars
  Amostra: {"meta": {"seed": 447782942, "persona": "satisfeito", "scenario": "Vazamento mas cliente fica satisf...

• Chamada 57 (Cliente: 57)
  Tipo: OUTRO | Tamanho: 18386 chars
  Amostra: {"meta": {"seed": 447783136, "persona": "irritado", "scenario": "Vazamento mas cliente fica satisfei...

• Chamada 64 (Cliente: 64)
  Tipo: OUTRO | Tamanho: 16031 chars
  Amostra: {"meta": {"seed": 447780323, "persona": "neu

In [31]:
# PROCESSAMENTO EM LOTE - ATUALIZAR NOTA_QUALIDADE E AVALIACAO_JSON
# =======================================================================
# VERSAO ATUALIZADA: Agora salva TANTO a nota quanto o JSON detalhado
# =======================================================================

def check_nota_qualidade_constraints():
    """
    Verifica as constraints da coluna nota_qualidade para entender os limites permitidos.
    """
    conn = get_db_connection()
    if not conn:
        print("Erro: Nao foi possivel conectar ao banco de dados")
        return None, None
    
    try:
        cursor = conn.cursor()
        
        # Verifica as constraints da tabela
        query_constraints = """
        SELECT conname, consrc 
        FROM pg_constraint 
        WHERE conrelid = 'chamada'::regclass 
        AND conname LIKE '%nota_qualidade%'
        """
        
        cursor.execute(query_constraints)
        constraints = cursor.fetchall()
        
        print("CONSTRAINTS DA COLUNA nota_qualidade:")
        
        for constraint_name, constraint_def in constraints:
            print(f"• {constraint_name}: {constraint_def}")
        
        # FORÇA A ESCALA 1-10 CONFORME SOLICITADO
        min_val, max_val = 1, 10
        print(f"Escala configurada manualmente: {min_val} a {max_val}")
        return min_val, max_val
        
    except Exception as e:
        print(f"Erro ao verificar constraints: {e}")
        return 1, 10  # padrao 1-10
    finally:
        cursor.close()
        conn.close()

def process_all_calls_with_detailed_json():
    """
    FUNCAO ATUALIZADA: Processa todas as chamadas e salva TANTO nota_qualidade quanto avaliacao_json detalhado.
    """
    # Primeiro, verifica as constraints
    min_nota, max_nota = check_nota_qualidade_constraints()
    
    conn = get_db_connection()
    if not conn:
        print("Erro: Nao foi possivel conectar ao banco de dados")
        return
    
    try:
        cursor = conn.cursor()
        
        # 1. Busca todas as chamadas que tem transcript_json
        print("\nBuscando todas as chamadas com transcricao...")
        query_select = """
        SELECT chamada_id, cliente_id, transcript_json, created_at, nota_qualidade, avaliacao_json
        FROM chamada 
        WHERE transcript_json IS NOT NULL
        ORDER BY chamada_id
        """
        
        cursor.execute(query_select)
        chamadas = cursor.fetchall()
        
        if not chamadas:
            print("Nenhuma chamada encontrada com transcricao")
            return
        
        print(f"Encontradas {len(chamadas)} chamadas para processar")
        print(f"Escala de notas: {min_nota} a {max_nota}")
        print("NOVIDADE: Salvando JSON detalhado em avaliacao_json")
        print("-" * 60)
        
        # Contadores para estatisticas
        processed_count = 0
        updated_count = 0
        error_count = 0
        
        # 2. Processa cada chamada
        for i, row in enumerate(chamadas, 1):
            chamada_id, cliente_id, transcript_json, created_at, nota_qualidade_atual, avaliacao_atual = row
            
            try:
                # Parse do JSON do transcript
                if isinstance(transcript_json, str):
                    transcript_data = json.loads(transcript_json)
                else:
                    transcript_data = transcript_json
                
                # Extrai o texto da transcricao
                transcript_text = extract_transcript_text(transcript_data)
                
                if not transcript_text or len(transcript_text.strip()) < 10:
                    print(f"Chamada {chamada_id}: Transcricao vazia ou muito curta - pulando")
                    error_count += 1
                    continue
                
                # Informacoes da chamada
                chamada_info = {
                    'chamada_id': chamada_id,
                    'cliente_id': cliente_id,
                    'created_at': created_at,
                    'nota_qualidade': nota_qualidade_atual
                }
                
                # USA A NOVA FUNCAO PARA AVALIACAO DETALHADA
                avaliacao_detalhada = evaluate_call_quality_detailed(transcript_text, chamada_info)
                
                # Extrai a nova nota calculada
                nova_nota = avaliacao_detalhada["resumo"]["nota_calculada_1_10"]
                
                # Garante que esta dentro dos limites 1-10
                nova_nota = max(min_nota, min(max_nota, nova_nota))
                
                # 3. Atualiza no banco TANTO a nota quanto o JSON
                query_update = """
                UPDATE chamada 
                SET nota_qualidade = %s, avaliacao_json = %s
                WHERE chamada_id = %s
                """
                
                cursor.execute(query_update, (
                    nova_nota,
                    json.dumps(avaliacao_detalhada, ensure_ascii=False),
                    chamada_id
                ))
                
                processed_count += 1
                
                # 4. Mostra progresso detalhado
                if i % 10 == 0 or i <= 20:
                    score_ponderado = avaliacao_detalhada["resumo"]["score_ponderado"]
                    criterios_ok = avaliacao_detalhada["resumo"]["criterios_aprovados"] 
                    total_criterios = avaliacao_detalhada["resumo"]["total_criterios"]
                    
                    print(f"Chamada {chamada_id}: Nota {nova_nota}/10 | Score {score_ponderado:.1%} | Criterios {criterios_ok}/{total_criterios}")
                    
                    # Mostra exemplo de criterios
                    if i <= 5:
                        print(f"    Exemplo de criterios salvos no JSON:")
                        for codigo, detalhes in list(avaliacao_detalhada["criterios"].items())[:3]:
                            resultado = detalhes["resultado"]
                            status = "OK" if resultado == "Aprovado" else "FALHOU"
                            print(f"       {status} {codigo}: {detalhes['descricao']} - {resultado}")
                
                conn.commit()
                updated_count += 1
                
            except Exception as e:
                print(f"Erro na chamada {chamada_id}: {e}")
                error_count += 1
                conn.rollback()
                continue
        
        # 5. Relatorio final detalhado
        print("\n" + "="*80)
        print("PROCESSAMENTO CONCLUIDO - VERSAO COM JSON DETALHADO!")
        print(f"Processadas com sucesso: {processed_count}")
        print(f"Atualizadas no banco: {updated_count}")
        print(f"Erros/Pulos: {error_count}")
        print(f"Taxa de sucesso: {(processed_count/len(chamadas)*100):.1f}%")
        
        # 6. Mostra algumas estatisticas das notas atualizadas
        print("\nESTATISTICAS DAS NOTAS CALCULADAS (ESCALA 1-10):")
        cursor.execute("""
        SELECT 
            COUNT(*) as total_chamadas,
            AVG(nota_qualidade) as nota_media,
            MIN(nota_qualidade) as nota_minima,
            MAX(nota_qualidade) as nota_maxima
        FROM chamada 
        WHERE nota_qualidade IS NOT NULL
        """)
        
        stats = cursor.fetchone()
        if stats:
            total, media, minima, maxima = stats
            print(f"• Total com nota: {total}")
            print(f"• Nota media: {media:.1f}/10")
            print(f"• Nota minima: {minima}/10") 
            print(f"• Nota maxima: {maxima}/10")
        
        # 7. Distribuicao das notas
        print("\nDISTRIBUICAO DAS NOTAS:")
        cursor.execute("""
        SELECT nota_qualidade, COUNT(*) as quantidade
        FROM chamada 
        WHERE nota_qualidade IS NOT NULL
        GROUP BY nota_qualidade
        ORDER BY nota_qualidade
        """)
        
        distribuicao = cursor.fetchall()
        for nota, quantidade in distribuicao:
            porcentagem = (quantidade / total * 100) if total > 0 else 0
            barra = "#" * max(1, int(porcentagem / 5))  # Escala visual
            print(f"  Nota {nota:2d}/10: {quantidade:3d} chamadas ({porcentagem:4.1f}%) {barra}")
        
        # 8. ESTATISTICAS DO JSON DETALHADO
        print("\nESTATISTICAS DO JSON DETALHADO:")
        cursor.execute("""
        SELECT COUNT(*) as total_com_json
        FROM chamada 
        WHERE avaliacao_json IS NOT NULL
        """)
        
        json_stats = cursor.fetchone()
        if json_stats:
            total_json = json_stats[0]
            print(f"• Chamadas com avaliacao JSON: {total_json}")
            print(f"• Estrutura salva: metadata, resumo, criterios, estatisticas_por_categoria")
            print(f"• Proximos passos: Usar APIs /api/v1/qualidade/* para consultar")
        
        print(f"\nCOMO CONSULTAR O JSON DE UMA CHAMADA:")
        print(f"   SELECT avaliacao_json FROM chamada WHERE chamada_id = [ID];")
        
        # 9. Mostra exemplo de JSON de uma chamada
        print(f"\nEXEMPLO DE JSON SALVO (chamada aleatoria):")
        cursor.execute("""
        SELECT chamada_id, avaliacao_json 
        FROM chamada 
        WHERE avaliacao_json IS NOT NULL 
        LIMIT 1
        """)
        
        exemplo = cursor.fetchone()
        if exemplo:
            exemplo_id, exemplo_json = exemplo
            print(f"Chamada {exemplo_id} (primeiros 300 caracteres):")
            json_str = json.dumps(exemplo_json, ensure_ascii=False, indent=2) if isinstance(exemplo_json, dict) else exemplo_json
            print(json_str[:300] + "...")
        
    except Exception as e:
        print(f"Erro geral no processamento: {e}")
        conn.rollback()
    finally:
        cursor.close()
        conn.close()

# FUNCAO PARA CONSULTAR DETALHES DE UMA CHAMADA ESPECIFICA
def query_chamada_json_details(chamada_id: int):
    """
    NOVA FUNCAO: Consulta e exibe os detalhes do JSON de avaliacao de uma chamada.
    """
    conn = get_db_connection()
    if not conn:
        print("Erro: Nao foi possivel conectar ao banco de dados")
        return
    
    try:
        cursor = conn.cursor()
        cursor.execute("""
        SELECT chamada_id, cliente_id, nota_qualidade, avaliacao_json, created_at
        FROM chamada 
        WHERE chamada_id = %s
        """, (chamada_id,))
        
        result = cursor.fetchone()
        if not result:
            print(f"Chamada {chamada_id} nao encontrada")
            return
        
        chamada_id, cliente_id, nota, avaliacao_json, created_at = result
        
        if not avaliacao_json:
            print(f"Chamada {chamada_id} nao possui avaliacao_json detalhado")
            return
        
        # Parse do JSON se necessario
        if isinstance(avaliacao_json, str):
            avaliacao_data = json.loads(avaliacao_json)
        else:
            avaliacao_data = avaliacao_json
        
        print(f"\nDETALHES DA AVALIACAO JSON - Chamada {chamada_id}")
        print("="*70)
        print(f"Cliente: {cliente_id} | Nota Final: {nota}/10 | Data: {created_at}")
        
        # Resumo
        resumo = avaliacao_data.get("resumo", {})
        print(f"\nRESUMO:")
        print(f"• Total de criterios: {resumo.get('total_criterios')}")
        print(f"• Criterios aprovados: {resumo.get('criterios_aprovados')}")
        print(f"• Score ponderado: {resumo.get('score_ponderado', 0):.1%}")
        print(f"• Nota calculada: {resumo.get('nota_calculada_1_10')}/10")
        
        # Criterios detalhados
        print(f"\nCRITERIOS DETALHADOS:")
        criterios = avaliacao_data.get("criterios", {})
        for codigo, detalhes in criterios.items():
            resultado = detalhes.get("resultado", "")
            descricao = detalhes.get("descricao", "")
            categoria = detalhes.get("categoria", "")
            peso = detalhes.get("peso", 1.0)
            evidencias = detalhes.get("evidencias_encontradas", [])
            
            status = "OK" if resultado == "Aprovado" else "FALHOU"
            print(f"{status} {codigo} ({categoria}) - Peso: {peso}")
            print(f"   {descricao}: {resultado}")
            if evidencias:
                print(f"   Evidencias: {', '.join(evidencias[:3])}")
        
        # Estatisticas por categoria
        print(f"\nESTATISTICAS POR CATEGORIA:")
        categorias = avaliacao_data.get("estatisticas_por_categoria", {})
        for cat, stats in categorias.items():
            taxa = stats.get("taxa_aprovacao", 0)
            aprovados = stats.get("aprovados", 0)
            total = stats.get("total", 0)
            print(f"• {cat}: {taxa:.1%} ({aprovados}/{total})")
        
        return avaliacao_data
        
    except Exception as e:
        print(f"Erro ao consultar detalhes: {e}")
        return None
    finally:
        cursor.close()
        conn.close()

# EXECUCAO DO PROCESSAMENTO
print("PROCESSAMENTO EM LOTE COM JSON DETALHADO - VERSAO 2.0")
print("="*70)
print("Esta operacao ira:")
print("1. Verificar constraints da coluna nota_qualidade")
print("2. Buscar todas as chamadas com transcricao")
print("3. Calcular o score de qualidade para cada uma")
print("4. Salvar nota_qualidade (1-10) E avaliacao_json detalhado")
print("5. Gerar estatisticas completas")
print("\nNOVO: JSON inclui evidencias, categorias, pesos e metadados!")
print("\nMAPEAMENTO DE NOTAS:")
print("• 0% de aprovacao (ponderado) = Nota 1")
print("• 50% de aprovacao (ponderado) = Nota 5-6") 
print("• 100% de aprovacao (ponderado) = Nota 10")
print("\nATENCAO: Esta operacao pode demorar alguns minutos")

print("\nIniciando processamento...")
process_all_calls_with_detailed_json()

print(f"\nPROXIMOS PASSOS:")
print(f"1. Use query_chamada_json_details(ID) para ver detalhes de uma chamada")
print(f"2. Inicie o backend: cd src/backend && npm run dev")
print(f"3. Teste as APIs: GET /api/v1/qualidade/insights-rapidos")

PROCESSAMENTO EM LOTE COM JSON DETALHADO - VERSAO 2.0
Esta operacao ira:
1. Verificar constraints da coluna nota_qualidade
2. Buscar todas as chamadas com transcricao
3. Calcular o score de qualidade para cada uma
4. Salvar nota_qualidade (1-10) E avaliacao_json detalhado
5. Gerar estatisticas completas

NOVO: JSON inclui evidencias, categorias, pesos e metadados!

MAPEAMENTO DE NOTAS:
• 0% de aprovacao (ponderado) = Nota 1
• 50% de aprovacao (ponderado) = Nota 5-6
• 100% de aprovacao (ponderado) = Nota 10

ATENCAO: Esta operacao pode demorar alguns minutos

Iniciando processamento...
Erro ao verificar constraints: column "consrc" does not exist
LINE 2:         SELECT conname, consrc 
                                ^
HINT:  Perhaps you meant to reference the column "pg_constraint.conkey" or the column "pg_constraint.conbin".

Erro ao verificar constraints: column "consrc" does not exist
LINE 2:         SELECT conname, consrc 
                                ^
HINT:  Perhaps you meant 

In [32]:
# 🆕 DEMONSTRAÇÃO: COMO USAR O JSON DETALHADO
# =============================================

# Exemplo 1: Consultar detalhes de uma chamada específica
print("📋 EXEMPLO 1: Consultando detalhes de uma chamada")
print("─" * 50)

# Busca uma chamada que tenha JSON para demonstrar
conn = get_db_connection()
if conn:
    cursor = conn.cursor()
    cursor.execute("""
    SELECT chamada_id FROM chamada 
    WHERE avaliacao_json IS NOT NULL 
    ORDER BY chamada_id 
    LIMIT 1
    """)
    result = cursor.fetchone()
    
    if result:
        exemplo_id = result[0]
        print(f"Chamada exemplo: {exemplo_id}")
        query_chamada_json_details(exemplo_id)
    else:
        print("❌ Nenhuma chamada com JSON encontrada")
        print("💡 Execute primeiro a célula de processamento em lote")
    
    cursor.close()
    conn.close()

print("\n" + "="*70)

# Exemplo 2: Consultas SQL úteis
print("📋 EXEMPLO 2: Consultas SQL úteis com o JSON")
print("─" * 50)

consultas_exemplo = [
    {
        "titulo": "🔍 Chamadas reprovadas em critério específico",
        "sql": """
        SELECT chamada_id, cliente_id, nota_qualidade
        FROM chamada 
        WHERE avaliacao_json->'criterios'->'2.1_empatia'->>'resultado' = 'Reprovado'
        LIMIT 5;
        """
    },
    {
        "titulo": "📊 Taxa de aprovação por categoria",
        "sql": """
        SELECT 
          categoria.key as categoria,
          AVG((categoria.value->>'taxa_aprovacao')::float) as media_aprovacao
        FROM chamada, 
             jsonb_each(avaliacao_json->'estatisticas_por_categoria') as categoria
        WHERE avaliacao_json IS NOT NULL
        GROUP BY categoria.key;
        """
    },
    {
        "titulo": "⚠️ Chamadas com baixo score em NCG",
        "sql": """
        SELECT chamada_id, cliente_id, nota_qualidade,
               avaliacao_json->'estatisticas_por_categoria'->'NCG / Falhas Graves'->>'taxa_aprovacao' as taxa_ncg
        FROM chamada 
        WHERE avaliacao_json->'estatisticas_por_categoria'->'NCG / Falhas Graves'->>'taxa_aprovacao' IS NOT NULL
          AND (avaliacao_json->'estatisticas_por_categoria'->'NCG / Falhas Graves'->>'taxa_aprovacao')::float < 0.7
        LIMIT 5;
        """
    }
]

for i, consulta in enumerate(consultas_exemplo, 1):
    print(f"\n{i}. {consulta['titulo']}:")
    print(consulta['sql'])

print("\n" + "="*70)

# Exemplo 3: APIs disponíveis
print("📋 EXEMPLO 3: APIs disponíveis no backend")
print("─" * 50)

apis_exemplo = [
    "GET /api/v1/qualidade/insights-rapidos?dias=7",
    "GET /api/v1/qualidade/estatisticas-por-criterio", 
    "GET /api/v1/qualidade/chamadas-baixa-qualidade?nota_minima=5",
    "GET /api/v1/qualidade/chamada/123/detalhes",
    "GET /api/v1/qualidade/chamadas-criterio-reprovado/2.1_empatia",
    "GET /api/v1/qualidade/resumo-periodo?dias=30"
]

for i, api in enumerate(apis_exemplo, 1):
    print(f"{i}. {api}")

print(f"\n💡 Para testar as APIs:")
print(f"1. cd src/backend")  
print(f"2. npm run dev")
print(f"3. curl http://localhost:4000/api/v1/qualidade/insights-rapidos")

print("\n🎉 MODELO COMPLETO COM JSON DETALHADO PRONTO!")

📋 EXEMPLO 1: Consultando detalhes de uma chamada
──────────────────────────────────────────────────
Chamada exemplo: 1

DETALHES DA AVALIACAO JSON - Chamada 1
Cliente: 1 | Nota Final: 5/10 | Data: 2025-09-12 13:48:41.582795

RESUMO:
• Total de criterios: 12
• Criterios aprovados: 5
• Score ponderado: 48.2%
• Nota calculada: 5/10

CRITERIOS DETALHADOS:
OK 4.0_etica (NCG / Falhas Graves) - Peso: 2.0
   Ética (não difamou a empresa ou colegas): Aprovado
OK 1.2_espera (Abertura/Encerramento) - Peso: 0.8
   Sinalizou procedimento de espera: Aprovado
   Evidencias: só um momento
OK 2.1_empatia (Condução do Atendimento) - Peso: 1.2
   Demonstrou empatia: Aprovado
   Evidencias: compreendo, sinto muito
FALHOU 1.1_abertura (Abertura/Encerramento) - Peso: 1.0
   Abertura de contato padrão: Reprovado
FALHOU 4.4_abertura_os (NCG / Falhas Graves) - Peso: 1.3
   Mencionou a abertura ou o registro de uma OS: Reprovado
FALHOU 1.3_encerramento (Abertura/Encerramento) - Peso: 1.0
   Encerramento de cont